# Amazon Sales Analytics

## Business Problem
How can an Indian e-commerce seller improve sales performance and order fulfilment by identifying high-performing product categories, regional demand patterns, and operational issues associated with cancellations and returns?

## Project Objectives
- Analyse sales performance across product categories.
- Identify regional sales patterns.
- Evaluate order fulfilment performance.
- Analyse cancellations and returns.
- Identify business insights and recommendations.

## Tools Used
- Python (Pandas, NumPy, Matplotlib, Seaborn)
- Excel
- MySQL
- Power BI

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

### 2. Load the dataset

In [3]:
current = Path.cwd()

project_root = next(
    (p for p in [current, *current.parents]
     if (p / "data" / "raw" / "Amazon Sale Report.csv").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "CSV not found. Check data/raw/Amazon Sale Report.csv"
    )

file_path = project_root / "data" / "raw" / "Amazon Sale Report.csv"

df = pd.read_csv(
    file_path,
    low_memory=False,
    dtype={"ship-postal-code": "string"}
)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)

df.head()

Dataset loaded successfully!
Rows and columns: (128975, 24)


,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,Size,ASIN,Courier Status,Qty,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,NaN,0,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,Easy Ship,NaN
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,NaN,NaN


## 4. Initial Data Understanding
Inspecting  the dataset structure, column names, data types, and basic statistical information before making any changes.

In [4]:
print("Dataset dimensions:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

Dataset dimensions: (128975, 24)

Column names:
['index', 'Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'ship-country', 'promotion-ids', 'B2B', 'fulfilled-by', 'Unnamed: 22']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  str    
 2   Date                128975 non-null  str    
 3   Status              128975 non-null  str    
 4   Fulfilment          128975 non-null  str    
 5   Sales Channel       128975 non-null  str    
 6   ship-service-level  128975 non-null  str    
 7   Style               128975 non-null  str    
 8   SKU                 12

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
index,128975.0,NaN,NaN,NaN,64487.0,37232.019822,0.0,32243.5,64487.0,96730.5,128974.0
Order ID,128975,120378,403-4984515-8861958,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date,128975,91,05-03-22,2085,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,128975,13,Shipped,77804,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fulfilment,128975,2,Amazon,89698,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sales Channel,128975,2,Amazon.in,128851,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ship-service-level,128975,2,Expedited,88615,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Style,128975,1377,JNE3797,4224,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SKU,128975,7195,JNE3797-KR-L,773,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Category,128975,9,Set,50284,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5. Checking Missing Values

In [5]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values("Missing Percentage", ascending=False)

display(missing_summary)

print("Exact duplicate rows:", df.duplicated().sum())
print("Unique Order IDs:", df["Order ID"].nunique())

,Missing Values,Missing Percentage
fulfilled-by,89698,69.55
promotion-ids,49153,38.11
Unnamed: 22,49050,38.03
currency,7795,6.04
Amount,7795,6.04
Courier Status,6872,5.33
ship-postal-code,33,0.03
ship-state,33,0.03
ship-city,33,0.03
ship-country,33,0.03


Exact duplicate rows: 0
Unique Order IDs: 120378


### 6. Remove Unnecessary Columns

Removing columns that do not contribute meaningful information to the business analysis.

- `index`: Sequential row number, not a business identifier.
- `Unnamed: 22`: Unnecessary empty column.
- `fulfilled-by`: Mostly missing and redundant with the fulfilment column.

The original CSV remains unchanged.

In [7]:
columns_to_drop = [
    "index",
    "Unnamed: 22",
    "fulfilled-by"
]

df = df.drop(
    columns=[col for col in columns_to_drop if col in df.columns]
)

print("Updated shape:", df.shape)
print("Remaining columns:")
print(df.columns.tolist())

Updated shape: (128975, 21)
Remaining columns:
['Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'Qty', 'currency', 'Amount', 'ship-city', 'ship-state', 'ship-postal-code', 'ship-country', 'promotion-ids', 'B2B']


## 7. Standardise Column Names

converting column names into lowercase snake_case format to make them easier to use in Python and SQL.

In [8]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-zA-Z0-9]+", "_", regex=True)
    .str.strip("_")
)

print(df.columns.tolist())

['order_id', 'date', 'status', 'fulfilment', 'sales_channel', 'ship_service_level', 'style', 'sku', 'category', 'size', 'asin', 'courier_status', 'qty', 'currency', 'amount', 'ship_city', 'ship_state', 'ship_postal_code', 'ship_country', 'promotion_ids', 'b2b']


## 8. Data Type Correction
Converting date and numerical columns into appropriate data types.

Postal codes and identifiers are kept as text because they are labels, not numerical measurements.

In [9]:
df["date"] = pd.to_datetime(
    df["date"],
    format="%m-%d-%y",
    errors="coerce"
)

df["qty"] = pd.to_numeric(
    df["qty"],
    errors="coerce"
)

df["amount"] = pd.to_numeric(
    df["amount"],
    errors="coerce"
)

text_columns = [
    "order_id",
    "sku",
    "asin",
    "ship_postal_code"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].astype("string")

print(df.dtypes)

order_id                      string
date                  datetime64[us]
status                           str
fulfilment                       str
sales_channel                    str
ship_service_level               str
style                            str
sku                           string
category                         str
size                             str
asin                          string
courier_status                   str
qty                            int64
currency                         str
amount                       float64
ship_city                        str
ship_state                       str
ship_postal_code              string
ship_country                     str
promotion_ids                    str
b2b                             bool
dtype: object


## 9. Text Standardisation

Removing unnecessary leading and trailing spaces from text columns to improve consistency during grouping and analysis.

In [10]:
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()

print("Text standardisation completed.")

Text standardisation completed.


### 10. Data Quality Validation

In [12]:
print("Dataset shape:", df.shape)

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
display(
    df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

print("\nQuantity distribution:")
display(df["qty"].describe())

print("\nAmount distribution:")
display(df["amount"].describe())

print("\nDate range:")
print(df["date"].min(), "to", df["date"].max())

Dataset shape: (128975, 21)

Exact duplicate rows: 6

Missing values:


,Missing Values
promotion_ids,49153
currency,7795
amount,7795
courier_status,6872
ship_city,33
ship_postal_code,33
ship_country,33
ship_state,33
sales_channel,0
order_id,0



Quantity distribution:


count    128975.000000
mean          0.904431
std           0.313354
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          15.000000
Name: qty, dtype: float64


Amount distribution:


count    121180.000000
mean        648.561465
std         281.211687
min           0.000000
25%         449.000000
50%         605.000000
75%         788.000000
max        5584.000000
Name: amount, dtype: float64


Date range:
2022-03-31 00:00:00 to 2022-06-29 00:00:00


## 11. Prepare Data for Analysis

Creating a few useful columns for analysing monthly sales and promotional activity.

In [14]:
df["order_month"] = df["date"].dt.strftime("%Y-%m")

df["has_promotion"] = df["promotion_ids"].notna()

print("Analysis columns created.")
display(df[["date", "order_month", "has_promotion"]].head())

Analysis columns created.


,date,order_month,has_promotion
0,2022-04-30,2022-04,False
1,2022-04-30,2022-04,True
2,2022-04-30,2022-04,True
3,2022-04-30,2022-04,False
4,2022-04-30,2022-04,False


## 12. Export Cleaned Dataset

Saving the cleaned dataset for further analysis using MySQL and Power BI.

In [17]:
output_folder = project_root / "data" / "processed"
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "amazon_sales_cleaned.csv"

df.to_csv(output_file, index=False)

print("Dataset exported successfully!")
print("Location:", output_file)
print("Final dimensions:", df.shape)

Dataset exported successfully!
Location: C:\Amazon Sales Analytics\data\processed\amazon_sales_cleaned.csv
Final dimensions: (128975, 23)


## 13. Final Validation

Verifying that the exported dataset is available and contains the expected records and columns.

In [18]:
cleaned_df = pd.read_csv(output_file, low_memory=False)

print("Exported dataset shape:", cleaned_df.shape)
print("Date range:", cleaned_df["date"].min(), "to", cleaned_df["date"].max())
print("Duplicate rows:", cleaned_df.duplicated().sum())

display(cleaned_df.head())

Exported dataset shape: (128975, 23)
Date range: 2022-03-31 to 2022-06-29
Duplicate rows: 6


,order_id,date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,asin,courier_status,qty,currency,amount,ship_city,ship_state,ship_postal_code,ship_country,promotion_ids,b2b,order_month,has_promotion
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,2022-04,False
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,2022-04,True
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,2022-04,True
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,NaN,0,INR,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,IN,NaN,False,2022-04,False
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,INR,574.00,CHENNAI,TAMIL NADU,600073.0,IN,NaN,False,2022-04,False


In [21]:
from pathlib import Path

new_file = file_path.parent / "amazon_sales_mysql_ready.csv"

df.to_csv(new_file, index=False)

print("New file saved:", new_file)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print(df["date"].head())

New file saved: C:\Amazon Sales Analytics\data\processed\amazon_sales_mysql_ready.csv
Rows: 128975
Columns: 23
0    2022-04-30
1    2022-04-30
2    2022-04-30
3    2022-04-30
4    2022-04-30
Name: date, dtype: str


In [8]:
import pandas as pd
from pathlib import Path

folder = Path(r"C:\Users\Jiyam\OneDrive\Desktop\Amazon Sales Analytics\data\processed")

source = folder / "amazon_sales_mysql_ready.csv"
output = folder / "amazon_sales_mysql_ready_v3.csv"

df = pd.read_csv(source)

print("Source rows:", len(df))

df.to_csv(output, index=False, chunksize=10000, lineterminator="\n")

# Verify the newly created file
check = pd.read_csv(output)

print("Export completed!")
print("Rows:", len(check))
print("Columns:", len(check.columns))
print("Missing amounts:", check["amount"].isna().sum())
print("File size (MB):", round(output.stat().st_size / (1024 * 1024), 2))

Source rows: 128975
Export completed!
Rows: 128975
Columns: 23
Missing amounts: 7795
File size (MB): 65.9
